# Journal de bord

## 📅 Journal de bord – Jour 1 (24/07/2026)
### 🔧 Étapes / Actions réalisées
- Découverte du sujet,
- Récupération des ressources,
- Structuration des dossiers
- Mise en place du projet VSCode
- Ajout du projet sur GitHub


## 📅 Journal de bord – Jour 2 (28/07/2026)
### 🔧 Étapes / Actions réalisées
- lecture et première analyse des données
- dictionnaire des données à réaliser 
- choix des colonnes à garder/supprimer et justification


### 💭 Réflexions, hypothèses et partis pris [C1][C2]

**Cadrage métier (chapitre 2)** : l'énoncé n'impose pas d'arbitrage entre faux positifs et faux
négatifs — c'est un choix politique/budgétaire propre à l'université, qui n'est pas précisé. Faute
de contrainte chiffrée, j'ai retenu l'hypothèse de **minimiser les faux négatifs** (favoriser le
rappel), car l'énoncé insiste explicitement sur le coût métier d'un décrochage non détecté. Ce
choix accepte plus de faux positifs (donc plus de ressources de tutorat mobilisées à tort) — un
compromis documenté mais non validé formellement par une "Direction" fictive, ce qui reste une
limite du cas d'usage.

**Disponibilité et gouvernance des données (chapitre 3)** : le jeu de données est volontairement
"brut" (précisé dans l'énoncé) — confirmé dès la première lecture : 40 doublons exacts, 3
variables **leurres** sans aucun pouvoir prédictif logique (`groupe_td`, `couleur_carte_etudiante`,
`jour_inscription`), présentes pour tester la rigueur méthodologique plutôt que pour enrichir le
modèle. Décision : les exclure explicitement plutôt que de laisser un algorithme de sélection de
variables les filtrer implicitement — je veux pouvoir justifier ce choix à l'oral.

**Enrichissement** : jointure avec `catalogue_formations_V5.csv` sur la colonne `filiere`, pour
récupérer faculté, capacité d'accueil et taux de réussite historique. Hypothèse : une filière
structurellement en difficulté (taux de réussite historique bas) porte un signal utile,
indépendamment du profil individuel de l'étudiant.

**Éthique et RGPD (chapitre 4)** : deux pièges de fuite de données identifiés dès la lecture du
dictionnaire —
1. `moyenne_finale` (l'autre cible) ne doit jamais expliquer `abandon` ;
2. `moyenne_partiels_s1` et `nb_ue_validees_s1` sont des résultats consolidés en **fin** de S1,
   donc indisponibles au moment du scoring à mi-S1 (fuite *temporelle*, pas seulement une fuite
   de cible).

Décision plus délicate : `sexe` et `boursier` sont identifiés comme variables sensibles. Rien
n'interdit *techniquement* de les utiliser, et elles ont probablement un pouvoir prédictif réel
(facteurs socio-économiques). J'ai choisi de les **exclure quand même** des features, par
précaution éthique (risque de biais discriminatoire et de renforcer des inégalités déjà connues),
en acceptant le compromis d'un modèle potentiellement un peu moins performant mais plus
défendable. Point à documenter clairement pour la soutenance.

## 📅 Journal de bord – Jour 3 et 4 (29,30/07/2026)
### 🔧 Étapes / Actions réalisées
- analyse exploratoire des données
- transformation des données dans le silver dataframe
- ajout/suppression de colonnes
- choix des méthodes pour remplir les valeurs nulles (NaN)
- génération du gold dataset

### 📝 Evénements/Difficultés

Il semble que le fait d'utiliser la médiane pour effectuer de l'imputation de données néccessite de le faire séparément entre Train et test. Le gold dataset n'est donc pas utilisable immédiatement.

### 💭 Réflexions, hypothèses et partis pris [C3]

**EDA (chapitre 6)** : la cible `abandon` est déséquilibrée (28,4 %) mais pas extrême — hypothèse
retenue : ce déséquilibre pourra être compensé par `class_weight="balanced"` au moment de
l'entraînement, sans recourir à du sur-échantillonnage (SMOTE), à valider une fois les premiers
modèles comparés (voir Jour 5). Le type de bac s'avère très discriminant (le bac professionnel
décroche nettement plus que le bac général) — confirme l'intérêt de nettoyer proprement cette
variable malgré ses encodages hétérogènes (majuscules, espaces).

**Préparation des données (chapitre 7)** : plusieurs choix structurants ici, chacun avec son
hypothèse :
- **Feature engineering** : `nb_devoirs_rendus` et `nb_devoirs_total` sont remplacés par un ratio
  `taux_rendu`. Hypothèse : un ratio est plus informatif qu'un simple compte (4 devoirs rendus sur
  5 n'a pas le même sens que 4 sur 13). Vérifié que le ratio ne dépasse jamais 1 avant de valider
  la substitution — sinon la colonne brute aurait dû être conservée en parallèle.
- **Imputation des valeurs manquantes** (jusqu'à 12 % sur certaines colonnes) : médiane retenue
  plutôt que suppression des lignes, pour ne pas perdre en taille d'échantillon (déjà limité à
  ~5 200 lignes). 
- **Encodage** : one-hot pour `filiere`/`bac_type`/`faculte`/`etablissement_origine` (catégories
  sans ordre naturel) vs encodage **ordinal** pour `mention_bac` (Passable < AB < Bien < TB — un
  vrai ordre qu'un one-hot aurait perdu).

**Difficulté majeure rencontrée** (déjà notée le jour même) : l'imputation par médiane doit être
calculée **séparément sur train et test** — `fit` sur le train uniquement, puis `transform` sur les
deux — pour ne pas laisser filtrer une information du test vers l'entraînement (fuite de
données). Conséquence concrète : le `gold_dataset` généré à cette étape n'est qu'un **produit
intermédiaire** ; le split train/test doit être fait *avant* l'imputation finale, pas après. Ce
principe anti-fuite est devenu une règle appliquée systématiquement pour le reste du projet.

## 📅 Journal de bord – Jour 5 (3/08/2026)
### 🔧 Étapes / Actions réalisées
- explicabilité SHAPE
- gestion des entrainements avec MLFlow
- nettoyage du projet
- choix final du modèle

### 💭 Réflexions, hypothèses et partis pris [C4][C5]

**Démarche scientifique (chapitre 8)** : posé une comparaison de 3 familles de modèles
(régression logistique, Random Forest, Gradient Boosting) plutôt qu'un choix a priori, avec
`class_weight="balanced"` pour compenser le déséquilibre de la cible identifié en EDA.

**Résultat qui a d'abord surpris** : c'est le modèle le plus simple, la régression logistique, qui
obtient le meilleur rappel — devant les 2 modèles d'arbres, réputés plus performants sur données
tabulaires. Hypothèse formulée pour l'expliquer, et confirmée plus tard par les coefficients et le
diagramme SHAP : la relation entre les variables d'engagement (taux de présence, connexions LMS)
et le risque de décrochage est **globalement linéaire** — un modèle linéaire capture bien ce
signal sans le sur-complexifier, contrairement à des arbres qui cherchent des interactions qui
n'existent pas vraiment ici.

**Validation croisée avant tuning** : une 5-fold CV sur les modèles par défaut a été faite en
premier, pour vérifier que l'écart entre modèles observé sur le seul split train/test n'est pas un
artefact de ce découpage précis. Les écarts-types obtenus étant faibles, le classement est jugé
fiable — condition posée avant de lancer l'optimisation des hyperparamètres.

**Tuning (chapitre 9)** : `GridSearchCV`/`RandomizedSearchCV` sur les 3 modèles, optimisé sur
ROC-AUC. Le classement ne change pas après tuning — confirme que la supériorité de la régression
logistique n'est pas un hasard de configuration par défaut. `C=0.01` retenu (forte régularisation),
ce qui a d'abord semblé contre-intuitif (peu de liberté laissée au modèle) mais s'explique par le
nombre de features assez élevé après one-hot encoding face à ~5 200 lignes.

**SHAP** : calculé pour vérifier la cohérence avec les coefficients et les corrélations déjà vues
en EDA — confirmé, `taux_presence_pct` est de loin la variable la plus déterminante. Ce
diagramme sert aussi d'argument d'explicabilité pour les équipes pédagogiques (exigence de
l'énoncé), pas seulement de contrôle interne.

**Choix du seuil de décision** : le seuil qui maximise le F1 (0,503) est presque identique au
seuil par défaut (0,5) — décision de garder 0,5 pour la simplicité, tout en documentant qu'un
seuil plus bas (~0,33-0,36) réduirait davantage les faux négatifs, au prix de plus de faux
positifs. Ce choix de seuil final est présenté comme un **arbitrage métier à valider avec la
Direction**, pas une décision purement technique — cohérent avec l'hypothèse retenue au Jour 2.

**MLflow** : mis en place pour tracer chaque run (baseline puis tuné, pour les 3 modèles) de façon
reproductible et comparable, avant de figer le modèle final dans le Model Registry.

## 📅 Journal de bord – Jour 6 (4/08/2026)
### 🔧 Étapes / Actions réalisées
- intégration d'un modèle linéaire pour prédiction de la moyenne

### 💭 Réflexions, hypothèses et partis pris [C5]

**Régression pour `moyenne_finale`** : même démarche scientifique que pour la classification —
comparaison de 3 familles, cette fois Ridge (baseline linéaire régularisée), Random Forest et
Gradient Boosting.

**Résultat inverse de la classification** : ici, c'est Gradient Boosting qui gagne, alors que
c'était la régression logistique (linéaire) qui gagnait côté classification. Hypothèse retenue
pour expliquer cette différence : la note finale continue dépend d'**interactions plus complexes**
entre variables (l'effet du taux de présence sur la note peut par exemple varier selon la
filière), que les arbres de décision capturent mieux qu'un modèle linéaire régularisé comme
Ridge. Les deux résultats ne se contredisent pas : ils reflètent simplement deux cibles aux
structures différentes.

**Hypothèse testée puis rejetée** : peut-on se contenter de dériver l'abandon à partir de la note
finale prédite (en seuillant `-moyenne_finale_predite`), plutôt que d'entraîner un classifieur
dédié ? Testé explicitement, avec la même méthodologie de seuillage — le classifieur direct reste
meilleur sur la quasi-totalité des indicateurs. Décision : garder les **deux modèles strictement
indépendants**, sans stacking ni fuite croisée entre les deux tâches, même si cela duplique une
partie de la préparation des données.

**Fonction combinée** : `predire_etudiant()` créée pour appeler les deux modèles ensemble côté
utilisation (une seule fonction, deux résultats), sans changer l'architecture de modélisation
sous-jacente — les deux modèles restent entraînés et évalués séparément.

## 📅 Journal de bord – Jour 7 (5/08/2026)
### 🔧 Étapes / Actions réalisées
- regroupement dans entrainement des 2 modèles (abandon et moyenne finale)
- création des fichiers de test pour l'intégration continue
- intégration dans docker incluant prometheus et graphana
- mise en place d'alertes de fonctionnement et de dérive.

### 💭 Réflexions, hypothèses et partis pris [C6]

**Consolidation** : regroupement des cellules d'entraînement final (jusque-là éparpillées) dans un
seul chapitre. Décision de lisibilité justifiée par une exigence explicite de l'énoncé : le
notebook doit être "exécutable et lisible sans explication orale".

**Tests automatisés** : constat, en voulant mettre en place l'intégration continue GitHub, qu'
**aucun test n'existait** dans le dépôt jusque-là — 49 tests créés a posteriori (package, config,
pipeline tabulaire, CLI, API, sécurité, stockage). Difficulté/leçon retenue : les tests auraient dû
être écrits en parallèle du code, pas après coup.

**API (chapitre 10)** : un seul endpoint `/predict-tabular` sert les deux modèles ensemble —
décision cohérente avec le choix "pas de stacking" du Jour 6 : chaque modèle est appelé
indépendamment sur la même ligne de features préparée, la mutualisation ne porte que sur le
contrat HTTP, pas sur la logique de prédiction.

**Monitoring** : au-delà des métriques HTTP génériques, ajout de métriques métier spécifiques
(`proba_abandon`, `moyenne_predite`, `model_loaded`) — nécessaires pour détecter une dérive
propre aux prédictions, pas seulement une panne technique. Les seuils des 3 alertes de dérive
ont été fixés **arbitrairement** (±10 points sur le taux d'abandon, ±3 points sur la note), faute
d'historique de production réel — limite documentée explicitement, à affiner avec de vraies
données une fois le service en exploitation. Testé réellement (pas seulement codé) : après
plusieurs appels avec le même profil d'étudiant, les alertes de dérive passent bien en
`pending` après quelques minutes, preuve que la détection fonctionne effectivement.

## 📅 Journal de bord – Jour 8 (6/08/2026)
### 🔧 Étapes / Actions réalisées
- Architecture cible et contraintes
- Mesure de performance et impacts (métriques techniques + métier)
- Amélioration continue (ré-entraînement, suivi, versioning)
- Conclusion (synthèse et recommandations)

### 💭 Réflexions, hypothèses et partis pris

**Architecture cible** : le constat principal de cette étape est que la brique manquante pour un
vrai service n'est **pas technique** (l'API fonctionnait déjà) mais la couche d'orchestration
applicative — sans stockage batch historisé, chaque consultation recalculerait tout et personne
ne pourrait suivre l'évolution du risque d'un étudiant dans le temps. Décision de l'implémenter
réellement (PostgreSQL + tâche Prefect dédiée) plutôt que de la laisser à l'état de recommandation.

**Mesure d'impact métier** : le coût d'un faux négatif (2 500 €, hypothèse fournie dans l'énoncé)
a été utilisé pour chiffrer concrètement l'enjeu — sur l'échantillon test, les faux négatifs non
détectés représentent 75 000 €, extrapolés à environ 375 000 €/an sur la promotion complète. Ce
chiffrage rend le choix du seuil (Jour 5) beaucoup plus concret pour un public non technique.

**Conclusion** : ce qui manque pour transformer le pilote en service exploité n'est
principalement pas technique, mais organisationnel — gouvernance humaine (qui valide un nouveau
modèle, qui arbitre le seuil, qui audite les accès).

## 📅 Journal de bord – Jour 9 (7/08/2026)
### 🔧 Étapes / Actions réalisées
- reformatage du notebook avec renumérotation des chapitres et ajout d'un sommaire cliquable
- discussion sur le seuil sur le base du calcul de ROI
- mise en place d'annexes

### 💭 Réflexions, hypothèses et partis pris

**Arbitrage du seuil, approfondi** : au-delà du choix initial (0,5 par défaut, Jour 5), un
chiffrage complet a été fait pour un seuil alternatif de 0,33. Résultat nuancé : abaisser le seuil
crée un **gain net supérieur** (+95 000 €/an environ), mais un **ROI (ratio bénéfice/coût)
inférieur** — car le nombre d'accompagnements déclenchés (donc leur coût) augmente
proportionnellement plus que les faux négatifs évités. Cette distinction entre gain net absolu et
ROI relatif a affiné la recommandation finale : les deux seuils sont défendables selon l'angle
retenu par la Direction, ce qui renforce l'idée que ce choix reste un arbitrage métier, pas une
simple optimisation technique.

## 📅 Journal de bord – Jour 10 (8/08/2026)
### 🔧 Étapes / Actions réalisées
- génération d'un pdf du notebook

## 📅 Journal de bord – Jour 11,12 (18/08/2026)
### 🔧 Étapes / Actions réalisées
- relecture/correction du notebook et notebook autonome